In [3]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


## Loading dataset

In [4]:
train_df = pd.read_csv('train.csv')

test_df = pd.read_csv('test.csv')

sample_df = pd.read_csv('sample_submission.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'test.csv'

In [ ]:
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Submission:", sample_df.shape)

In [ ]:
train_df.dtypes.value_counts()

In [ ]:
train_df.select_dtypes(include='object').columns.tolist()

In [ ]:

print("Activity distribution:")
print(train_df['target'].value_counts())


print("\nFirst 5 sensor readings:")
print(train_df['t000_chest_acc_x'].head())


print("\nSensor column datatype:")
print(train_df['t000_chest_acc_x'].dtype)


sensor_col = 't000_chest_acc_x'

converted = pd.to_numeric(train_df[sensor_col], errors='coerce')

print("\nNumber of non-numeric values:")
print(converted.isna().sum())

In [ ]:
sensor_col = 't000_chest_acc_x'


print("Existing missing values:")
print(train_df[sensor_col].isna().sum())


converted = pd.to_numeric(train_df[sensor_col], errors='coerce')


invalid_mask = train_df[sensor_col].notna() & converted.isna()

print("\nActual non-numeric values:")
print(invalid_mask.sum())


print("\nProblematic entries:")
print(train_df.loc[invalid_mask, sensor_col].head(20))


print("\nOriginal values and their Python types:")
print(train_df[sensor_col].map(type).value_counts())

In [ ]:

print("Data types:")
print(train_df.dtypes.value_counts())


print("\nTotal missing values:")
print(train_df.isnull().sum().sum())


missing_cols = train_df.columns[train_df.isnull().any()]

print("\nNumber of columns with missing values:")
print(len(missing_cols))

print("\nMissing values by column:")
print(train_df[missing_cols].isnull().sum().head(20))

In [ ]:

print("Missing ID values:", train_df['id'].isna().sum())
print("Missing target values:", train_df['target'].isna().sum())


row_missing = train_df.iloc[:, 1:-1].isnull().sum(axis=1)

print("\nMissing values per row:")
print(row_missing.describe())

print("\nTarget datatype:")
print(train_df['target'].dtype)


print("\nUnique target values:")
print(train_df['target'].unique())

In [ ]:


plt.figure(figsize=(12, 5))

sns.countplot(
    data=train_df,
    x='target',
    order=train_df['target'].value_counts().index
)

plt.title('Distribution of Physical Activities')
plt.xlabel('Activity')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### Observations – Activity Distribution

- The dataset contains 12 physical activity classes.
- Cycling has the highest number of samples (67).
- Jump front back has the lowest number of samples (19).
- The dataset exhibits class imbalance, as the number of samples varies across activity classes.
- This imbalance will be considered during model evaluation and selection.

####  Sensor Feature Analysis

Each activity sample contains multivariate time-series data collected from wearable sensors. We will examine the number of sensor features, their naming conventions, and their ordering.

In [ ]:

print("Training columns:", train_df.shape[1])
print("Testing columns:", test_df.shape[1])


print("Sensor features:", train_df.shape[1] - 2)

In [ ]:

print(train_df.columns[:30].tolist())


print(train_df.columns[-30:].tolist())

In [ ]:

feature_cols = train_df.columns[1:-1]


channels = pd.Series(feature_cols).str.replace(
    r'^t\d+_', '', regex=True
).unique()


timestamps = pd.Series(feature_cols).str.extract(
    r'^(t\d+)_'
)[0].unique()

print("Number of sensor channels:", len(channels))
print("\nSensor channels:")
print(channels)

print("\nNumber of time samples:", len(timestamps))
print("\nFirst timestamp:", timestamps[0])
print("Last timestamp:", timestamps[-1])


features_per_time = pd.Series(feature_cols).str.extract(
    r'^(t\d+)_'
)[0].value_counts()

print("\nFeatures per timestamp:")
print(features_per_time.value_counts())

### Observations – Sensor Feature Structure

- The dataset contains 5,750 sensor features.
- Each activity sample consists of 250 time samples.
- There are 23 sensor channels at each timestamp.
- The sampling frequency is 50 Hz.
- Each activity window represents 5 seconds of sensor data.
- The features follow a time-major naming convention, ranging from t000 to t249.

The feature structure is consistent with the expected dimensions of the multivariate time-series dataset.

##  Data Quality Analysis

In this section, we investigate missing values, inconsistent data representations, non-numeric entries and potentially unusual sensor readings to understand the quality of the dataset before preprocessing.

In [ ]:

sensor_data = train_df.iloc[:, 1:-1]


print("Existing missing values:",
      sensor_data.isna().sum().sum())


print("\nOccurrences of unknown:")
print(sensor_data.eq('unknown').sum().sum())

print("\nOccurrences of question marks:")
print(sensor_data.eq('?').sum().sum())

print("\nOccurrences of hyphens:")
print(sensor_data.eq('-').sum().sum())


comma_count = sensor_data.apply(
    lambda col: col.str.contains(',', na=False).sum()
).sum()

print("\nValues containing decimal commas:", comma_count)

In [ ]:

print("Examples of decimal-comma values:")

comma_values = sensor_data.apply(
    lambda col: col[col.str.contains(',', na=False)]
)

print(comma_values.stack().head(15))


print("\nOccurrences of 9999:")

print(
    sensor_data.apply(
        lambda col: col.str.contains('9999', na=False).sum()
    ).sum()
)



print("\nUnique suspicious text values:")

print(
    pd.unique(
        sensor_data.values.ravel()
    )[:30]
)

### Investigation of Unusual Sensor Values

We inspect frequently occurring values and suspicious numerical representations to distinguish valid sensor measurements from potential missing-value indicators and invalid entries.

In [ ]:

all_values = sensor_data.astype("string").stack()


all_values = all_values.str.strip()

print("Exact 9999 occurrences:")

print(
    all_values.isin([
        '9999',
        '9999.0',
        '9999,00',
        '9999.00'
    ]).sum()
)


print("\nMost common values:")
print(all_values.value_counts().head(20))


print("\nPlaceholder frequencies:")

for value in ['unknown', '?', '-']:
    print(value, ":", (all_values == value).sum())

In [ ]:


sentinel_values = ['9999', '9999.0', '9999,00', '9999.00']

sentinel_counts = sensor_data.astype('string').apply(
    lambda col: col.str.strip().isin(sentinel_values).sum()
)


print("Top 20 columns containing 9999:")

print(sentinel_counts.sort_values(
    ascending=False
).head(20))


print("\nTotal 9999 representations:")
print(sentinel_counts.sum())

In [ ]:


print("Total sensor columns:", len(sentinel_counts))

print("Columns containing 9999:",
      (sentinel_counts > 0).sum())

print("Columns without 9999:",
      (sentinel_counts == 0).sum())

print("\nStatistics of 9999 occurrences per column:")
print(sentinel_counts.describe())


plt.figure(figsize=(10, 5))

sns.histplot(sentinel_counts, bins=20)

plt.title('Distribution of 9999 Values Across Sensor Features')
plt.xlabel('Number of 9999 occurrences per column')
plt.ylabel('Number of sensor columns')

plt.tight_layout()
plt.show()

In [ ]:

sentinel_values = ['9999', '9999.0', '9999,00', '9999.00']


sentinel_per_row = sensor_data.astype('string').apply(
    lambda col: col.str.strip().isin(sentinel_values)
).sum(axis=1)


print("Sentinel values per row:")
print(sentinel_per_row.describe())


print("\nRows containing at least one sentinel:")
print((sentinel_per_row > 0).sum())


print("\nMaximum sentinels in one row:")
print(sentinel_per_row.max())

### Observations – Row-wise Sentinel Analysis

- All 679 training samples contain at least one occurrence of the value 9999.
- The average number of sentinel representations per sample is approximately 255.
- The minimum number per sample is 204.
- The maximum number per sample is 311.
- Sentinel candidates are present throughout the training dataset rather than being restricted to particular samples.

These findings indicate that the potential sentinel values must be considered during data preprocessing.

### Comprehensive Missing and Invalid Value Analysis

We examine all identified missing-value indicators together to understand the extent of incomplete or invalid sensor measurements before preprocessing.


In [ ]:
# Convert sensor data into string format
sensor_str = sensor_data.astype('string').apply(
    lambda col: col.str.strip()
)

# Identify missing and invalid entries
missing_mask = sensor_str.isna()

invalid_mask = sensor_str.isin([
    'unknown', '?', '-',
    '9999', '9999.0', '9999,00', '9999.00'
])

# Combine both masks
problem_mask = missing_mask | invalid_mask

print("Existing NaN values:",
      missing_mask.sum().sum())

print("Invalid/sentinel entries:",
      invalid_mask.sum().sum())

print("Total problematic entries:",
      problem_mask.sum().sum())

print("\nPercentage of problematic entries:")
print(
    problem_mask.sum().sum()
    / sensor_data.size * 100
)

### Observations – Comprehensive Data Quality Analysis

- The training dataset contains 3,904,250 sensor entries.
- A total of 429,835 entries are already missing (NaN).
- Another 209,007 entries contain invalid placeholders or potential sentinel values.
- Overall, approximately 16.36% of sensor entries are identified as problematic.
- These entries require appropriate preprocessing before model development.

### Test Dataset Quality Investigation

We examine the test dataset for missing values, invalid placeholders and inconsistent numerical representations to understand its data quality before applying preprocessing.

In [ ]:

test_sensor_data = test_df.iloc[:, 1:]


test_sensor_str = test_sensor_data.astype('string').apply(
    lambda col: col.str.strip()
)


test_missing = test_sensor_str.isna().sum().sum()

test_invalid = test_sensor_str.isin([
    'unknown', '?', '-',
    '9999', '9999.0', '9999,00', '9999.00'
]).sum().sum()


test_problematic = test_missing + test_invalid

print("Test sensor entries:", test_sensor_data.size)
print("Missing values:", test_missing)
print("Invalid/sentinel entries:", test_invalid)
print("Total problematic entries:", test_problematic)

print("\nPercentage of problematic entries:")
print(test_problematic / test_sensor_data.size * 100)

In [ ]:

train_features = train_df.columns[1:-1]
test_features = test_df.columns[1:]

print("Same feature names:",
      train_features.equals(test_features))

print("\nTraining data types:")
print(train_df.iloc[:, 1:-1].dtypes.value_counts())

print("\nTesting data types:")
print(test_df.iloc[:, 1:].dtypes.value_counts())

print("\nFirst 10 test sensor readings:")
print(test_df.iloc[:5, 1:6])

### Observations – Training and Testing Dataset Comparison

- Both datasets contain 5,750 sensor features with identical column names and ordering.
- Training sensor features have object datatype due to inconsistent data representations.
- Testing sensor features are already represented as float64.
- No missing values or specified invalid placeholders were detected in the test dataset using our checks.
- Consistent preprocessing will be required to ensure compatibility between training and testing data.

## Data Cleaning & Preprocessing

In this section, we clean the training dataset by:
1. Converting decimal commas (e.g., `'20,93'`) to standard dots (`.`).
2. Replacing string placeholders (`'unknown'`, `'?'`, `'-'`) and sentinel values (`'9999'`, `'9999.0'`, `'9999,00'`, `'9999.00'`) with `NaN`.
3. Converting all sensor columns to numeric `float64` data type.
4. Performing linear interpolation across time steps per sensor channel to handle missing values.

In [ ]:

train_clean = train_df.copy()
test_clean = test_df.copy()


feature_cols = train_df.columns[1:-1].tolist()

print("Cleaning training dataset values...")


invalid_sentinels = ['unknown', '?', '-', '9999', '9999.0', '9999,00', '9999.00', 'nan', 'None']

for col in feature_cols:
    if train_clean[col].dtype == 'object':
      
        train_clean[col] = train_clean[col].astype(str).str.replace(',', '.', regex=False).str.strip()
       
        train_clean[col] = train_clean[col].replace(invalid_sentinels, np.nan)
      
        train_clean[col] = pd.to_numeric(train_clean[col], errors='coerce')

print("Data types after cleaning:", train_clean[feature_cols].dtypes.value_counts())
print("Total NaN values before interpolation:", train_clean[feature_cols].isna().sum().sum())

### 2. Time-Series Interpolation & 3D Reshaping

Since sensor windows contain continuous time-series data, we perform linear interpolation along the time steps (250 time samples) for each sensor channel (23 channels). 
After interpolation, we reshape the 2D tabular data `(N, 5750)` into a 3D matrix `(N, 250, 23)` representing `(Samples, Time Steps, Channels)`.

In [ ]:

n_samples_train = len(train_clean)
n_samples_test = len(test_clean)
n_timesteps = 250
n_channels = 23

X_train_raw = train_clean[feature_cols].values
X_train_3d = X_train_raw.reshape(n_samples_train, n_timesteps, n_channels)


X_test_raw = test_clean[feature_cols].values.astype(np.float64)
X_test_3d = X_test_raw.reshape(n_samples_test, n_timesteps, n_channels)

print("Interpolating missing values across continuous time steps...")


for i in range(n_samples_train):
    sample_df = pd.DataFrame(X_train_3d[i])
  
    sample_df = sample_df.interpolate(method='linear', limit_direction='both', axis=0)
   
    sample_df = sample_df.bfill().ffill().fillna(0.0)
    X_train_3d[i] = sample_df.values

print("Training 3D shape:", X_train_3d.shape)
print("Testing 3D shape:", X_test_3d.shape)
print("Remaining NaNs in Train dataset:", np.isnan(X_train_3d).sum())
print("Remaining NaNs in Test dataset:", np.isnan(X_test_3d).sum())

## Feature Extraction from Time-Series Windows

To overcome high-dimensionality ($5,750$ features vs $679$ samples), we extract domain-specific summary statistics across the 250 time steps for each of the 23 sensor channels:
1. **Time-domain features:** Mean, Standard Deviation, Min, Max, Range, Median, Interquartile Range (IQR), Skewness, Kurtosis, Energy, RMS, Zero-Crossing Rate.
2. **Frequency-domain (FFT) features:** Mean Spectral Amplitude, Peak Spectral Amplitude.

This reduces our feature space from 5,750 raw readings down to 322 meaningful statistical features ($23 \times 14$).

In [ ]:
from scipy.stats import skew, kurtosis

def extract_features_from_3d(X_3d):
    """
    Computes statistical features across the 250 time steps for each of the 23 channels.
    Input shape:  (N_samples, 250, 23)
    Output shape: (N_samples, 23 * 14) -> (N_samples, 322)
    """
    n_samples, n_steps, n_channels = X_3d.shape
    extracted_rows = []

    for i in range(n_samples):
        sample = X_3d[i]  # shape: (250, 23)
        row_feats = []

        for ch in range(n_channels):
            signal = sample[:, ch]

            mean_val = np.mean(signal)
            std_val = np.std(signal)
            min_val = np.min(signal)
            max_val = np.max(signal)
            range_val = max_val - min_val
            median_val = np.median(signal)
            iqr_val = np.percentile(signal, 75) - np.percentile(signal, 25)
            skew_val = skew(signal)
            kurt_val = kurtosis(signal)
            energy_val = np.sum(signal**2) / n_steps
            rms_val = np.sqrt(energy_val)
            zcr_val = np.sum(np.diff(np.sign(signal)) != 0) / n_steps

            fft_vals = np.abs(np.fft.rfft(signal))
            fft_mean = np.mean(fft_vals)
            fft_max = np.max(fft_vals)

        
            ch_metrics = [
                mean_val, std_val, min_val, max_val, range_val,
                median_val, iqr_val, skew_val, kurt_val, energy_val,
                rms_val, zcr_val, fft_mean, fft_max
            ]
            row_feats.extend(ch_metrics)

        extracted_rows.append(row_feats)

    return np.array(extracted_rows)

print("Extracting features from 3D train dataset...")
X_train_extracted = extract_features_from_3d(X_train_3d)

print("Extracting features from 3D test dataset...")
X_test_extracted = extract_features_from_3d(X_test_3d)

print("Extracted Train Feature Shape:", X_train_extracted.shape)
print("Extracted Test Feature Shape:", X_test_extracted.shape)

## Model Selection and Evaluation

After preprocessing and feature extraction, each activity sample is represented by 322 extracted features.

In this section, we will train and evaluate machine learning classifiers using Stratified 5-Fold Cross-Validation.

Stratified K-Fold maintains approximately the same class distribution across all folds. This is useful because our dataset has an imbalanced activity distribution.

The evaluation metric is classification accuracy, as specified in the project instructions.

Since subject identifiers are unavailable, subject-wise validation is not possible. We therefore use stratified cross-validation as an alternative validation strategy, while acknowledging that it may not fully represent generalization to unseen subjects.

In [ ]:


print("Training shape:", X_train_extracted.shape)
print("Testing shape:", X_test_extracted.shape)

print("\nMissing values:")
print("Train:", np.isnan(X_train_extracted).sum())
print("Test:", np.isnan(X_test_extracted).sum())

print("\nInfinite values:")
print("Train:", np.isinf(X_train_extracted).sum())
print("Test:", np.isinf(X_test_extracted).sum())

print("\nData types:")
print("Train:", X_train_extracted.dtype)
print("Test:", X_test_extracted.dtype)

print("\nNumber of classes:", train_df['target'].nunique())
print("Classes:", train_df['target'].unique())


## 6. Model Training and Evaluation

In this section, different machine learning classifiers
are trained and compared using Stratified 5-Fold
Cross-Validation.

The objective is to identify the best-performing
classifier for physical activity recognition.


In [ ]:


X = X_train_extracted
y = train_df['target']

print("Training features shape:", X.shape)
print("Target labels shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


rf_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Random Forest Fold Accuracies:", rf_scores)
print("Mean CV Accuracy:", rf_scores.mean())
print("Standard Deviation:", rf_scores.std())


In [ ]:
from sklearn.utils import shuffle

y_shuffled = shuffle(y, random_state=42)

shuffled_scores = cross_val_score(
    rf_model,
    X,
    y_shuffled,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Shuffled-label accuracies:", shuffled_scores)
print("Mean:", shuffled_scores.mean())

In [ ]:

from sklearn.ensemble import ExtraTreesClassifier


et_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)


et_scores = cross_val_score(
    et_model,
    X,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Extra Trees Fold Accuracies:", et_scores)
print("Mean CV Accuracy:", et_scores.mean())
print("Standard Deviation:", et_scores.std())


In [ ]:

from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


svm_model = make_pipeline(
    StandardScaler(),
    SVC(
        kernel='rbf',
        C=10,
        gamma='scale',
        class_weight='balanced'
    )
)


svm_scores = cross_val_score(
    svm_model,
    X,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("SVM Fold Accuracies:", svm_scores)
print("Mean CV Accuracy:", svm_scores.mean())
print("Standard Deviation:", svm_scores.std())


In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Random Forest', 'Extra Trees', 'SVM'],
    'Mean CV Accuracy': [
        rf_scores.mean(),
        et_scores.mean(),
        svm_scores.mean()
    ],
    'Std Deviation': [
        rf_scores.std(),
        et_scores.std(),
        svm_scores.std()
    ]
})

print(results.sort_values('Mean CV Accuracy', ascending=False))

In [ ]:
rf_model.fit(X, y)

test_predictions = rf_model.predict(X_test_extracted)

print("Test predictions shape:", test_predictions.shape)
print("Predicted classes:", sorted(set(test_predictions)))
print("Number of predictions:", len(test_predictions))

In [ ]:
print("Sample submission shape:", sample_df.shape)
print("Test predictions length:", len(test_predictions))
print("Test dataframe shape:", test_df.shape)
print("Sample submission columns:", sample_df.columns.tolist())

In [ ]:
import pandas as pd

print("CHECKING SUBMISSION TEMPLATE NOW")

sample_check = pd.read_csv('sample_submission.csv')

print("Shape:", sample_check.shape)
print("Columns:", sample_check.columns.tolist())
print(sample_check.head())

In [ ]:
submission = sample_check.copy()

submission['target'] = test_predictions

print(submission.head())
print("Submission shape:", submission.shape)
print("Missing values:", submission.isnull().sum().sum())

submission.to_csv('submission.csv', index=False)

print("SUCCESS! submission.csv created.")

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('et', et_model)
    ],
    voting='soft',
    n_jobs=-1
)

ensemble_scores = cross_val_score(
    ensemble_model, X, y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Ensemble fold accuracies:", ensemble_scores)
print("Ensemble mean accuracy:", ensemble_scores.mean())
print("Ensemble standard deviation:", ensemble_scores.std())

In [ ]:

ensemble_model.fit(X, y)


ensemble_predictions = ensemble_model.predict_proba(X_test_extracted)

ensemble_predictions = ensemble_model.classes_[
    ensemble_predictions.argmax(axis=1)
]

print("Predictions shape:", ensemble_predictions.shape)
print("Number of predictions:", len(ensemble_predictions))

In [ ]:
submission_ensemble = sample_check.copy()

submission_ensemble['target'] = ensemble_predictions

print(submission_ensemble.head())
print("Shape:", submission_ensemble.shape)
print("Missing values:", submission_ensemble.isnull().sum().sum())

submission_ensemble.to_csv(
    'submission_ensemble.csv',
    index=False
)

print("SUCCESS! Second submission saved.")